# Deploy to an online endpoint

To use a model from an application, you can deploy the model to an online endpoint. You'll create an MLflow model from local files and test the endpoint.

## Before you start

You'll need the latest version of the **azureml-ai-ml** package to run the code in this notebook. Run the cell below to verify that it is installed.

> **Note**:
> If the **azure-ai-ml** package is not installed, run `pip install azure-ai-ml` to install it.

## Connect to your workspace

With the required SDK packages installed, you can now connect to your workspace.

To connect to a workspace, you need identifier parameters: a subscription ID, resource group name, and workspace name. The resource group name and workspace name are already filled in. You only need the subscription ID to complete the command.

To find the required parameters, click the subscription and workspace name in the upper right of the Studio. A pane will open on the right.

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> Copy the subscription ID and replace <strong>YOUR-SUBSCRIPTION-ID</strong> with the copied value. </p>

## Define and create an endpoint

The goal is to deploy a model to an endpoint. To do this, you first need to create the endpoint. The endpoint will be an HTTPS endpoint that applications can call to receive predictions from the model. An application can use its URI and authenticate with a key or token to use the endpoint.

Run the following cell to define the endpoint. Note that the name of the endpoint must be unique. You'll use the `datetime` function to generate a unique name.

Then, run the following cell to create the endpoint. This may take a few minutes. While the endpoint is being created, you can read about [what are Azure Machine Learning endpoints](https://learn.microsoft.com/azure/machine-learning/concept-endpoints).

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> IMPORTANT! Wait until the endpoint is successfully created before continuing! A green notification should appear in the studio. </p>

## Configure the deployment

You can deploy multiple models to an endpoint. This is most useful when you want to update a deployed model while keeping the current model in production. You'll need to configure the deployment to specify which model should be deployed to the endpoint. In the following cell, you'll refer to the model trained and stored in the local `model` folder (saved in the same folder as this notebook). Since you're working with an MLflow model, you don't need to specify the environment or a scoring script.

You'll also specify the infrastructure needed to deploy the model.

## Create the deployment

Finally, you can run the following cell to actually deploy the model to the endpoint.

In [ ]:
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

The deployment of the model may take 10-15 minutes. While waiting for the model to be deployed, you can learn more about [managed endpoints in this video](https://www.youtube.com/watch?v=SxFGw_OBxNM&ab_channel=MicrosoftDeveloper).

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> IMPORTANT! Wait until the deployment is completed before continuing! A green notification should appear in the studio.</p>

Since you only have one model deployed to the endpoint, you want this deployment to take 100% of the traffic. If you deploy multiple models to the endpoint, you could use the same approach to distribute traffic across the deployed models.

In [ ]:
# blue deployment takes 100 traffic
endpoint.traffic = {"blue": 100}
ml_client.begin_create_or_update(endpoint).result()

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> IMPORTANT! Wait until the blue deployment is configured before continuing! A green notification should appear in the studio. </p> 

## Test the deployment

Let's test the deployed model by invoking the endpoint. A JSON file with sample data is used as input. The trained model predicts whether a patient has diabetes or not, based on medical data like age, BMI, and the number of pregnancies. A `[0]` indicates a patient doesn't have diabetes. A `[1]` means a patient does have diabetes.

In [ ]:
# test the blue deployment with some sample data
response = ml_client.online_endpoints.invoke(
    endpoint_name=online_endpoint_name,
    deployment_name="blue",
    request_file="sample-data.json",
)

if response[1]=='1':
    print("Diabetic")
else:
    print ("Not diabetic")

Optionally, you can change the values in the `sample-data.json` file to try and get a different prediction.

## List endpoints

Although you can view all endpoints in the Studio, you can also list all endpoints using the SDK:

In [ ]:
endpoints = ml_client.online_endpoints.list()
for endp in endpoints:
    print(endp.name)

## Get endpoint details

If you want more information about a specific endpoint, you can explore the details using the SDK too.

In [ ]:
# Get the details for online endpoint
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

# existing traffic details
print(endpoint.traffic)

# Get the scoring URI
print(endpoint.scoring_uri)

## Delete the endpoint and deployment

As an endpoint is always available, it can't be paused to save costs. To avoid unnecessary costs, delete the endpoint.

In [ ]:
ml_client.online_endpoints.begin_delete(name=online_endpoint_name)